# Extraction et traitement de la FAQ

In [1]:
# Import des modules nécessaires
! pip install pdfplumber

from datasets import load_dataset
import pdfplumber
import re
import json

In [2]:
# Monter google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Extraire la FAQ
import pdfplumber

def extract_pdf_text(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

pdf_path = "drive/MyDrive/data_projet_llm/data/faq/FAQ_Hotel.pdf"
faq_text = extract_pdf_text(pdf_path)

print(faq_text[:1000])

FAQ — Hôtel De la Promenade
Un nombre limité de questions vous sont proposées, il est recommandé de rajouter des exemples pour
bonifier votre jeu de données.
Q1 : À quelle heure puis-je m'enregistrer à l'hôtel ?
R : Bienvenue à l'Hôtel De la Promenade ! Notre enregistrement standard débute à 15h00, moment où votre
refuge urbain vous attend, fraîchement préparé. Vous arrivez plus tôt et souhaitez déposer vos valises ?
Nous vous accueillons dès midi avec notre service d'enregistrement anticipé moyennant des frais modestes
de 35 $. Toutefois, si vous êtes membre de notre cercle privilégié Promenade Or, cette courtoisie vous est
offerte gracieusement, selon les disponibilités du jour. Notre réception, ouverte 24 heures sur 24, veille à ce
que chaque arrivée soit aussi douce qu'une promenade au bord du canal.
Q2 : Puis-je amener mon chien à l'hôtel ?
R : Mais certainement ! Chez nous, les compagnons à quatre pattes font partie de la famille. Nous
accueillons vos fidèles amis moyennant des f

In [4]:
# Pretraiter la FAQ
import re, json

def normalize_spaces(s: str) -> str:
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def extract_qa_pairs(faq_text: str):
    # enlever balises éventuelles
    faq_text = re.sub(r"<!--.*?-->", "", faq_text, flags=re.S)

    # regex robuste
    pattern = re.compile(r"Q(\d+)\s*:\s*(.*?)\s*R\s*:\s*(.*?)(?=\s*Q\d+\s*:|\Z)", re.S)
    pairs = []
    for m in pattern.finditer(faq_text):
        qnum = int(m.group(1))
        q = normalize_spaces(m.group(2))
        a = normalize_spaces(m.group(3))
        pairs.append((qnum, q, a))
    return pairs

In [5]:
pairs = extract_qa_pairs(faq_text)
print("Nombre:", len(pairs))
print(pairs[0])

Nombre: 30
(1, "À quelle heure puis-je m'enregistrer à l'hôtel ?", "Bienvenue à l'Hôtel De la Promenade ! Notre enregistrement standard débute à 15h00, moment où votre refuge urbain vous attend, fraîchement préparé. Vous arrivez plus tôt et souhaitez déposer vos valises ? Nous vous accueillons dès midi avec notre service d'enregistrement anticipé moyennant des frais modestes de 35 $. Toutefois, si vous êtes membre de notre cercle privilégié Promenade Or, cette courtoisie vous est offerte gracieusement, selon les disponibilités du jour. Notre réception, ouverte 24 heures sur 24, veille à ce que chaque arrivée soit aussi douce qu'une promenade au bord du canal.")


In [6]:
# Definir le modele de chat de Llama
SYSTEM = (
    "Vous êtes l’assistant officiel de l’Hôtel De la Promenade. "
    "Répondez en français, avec un ton chaleureux, élégant et professionnel. "
    "Donnez des informations exactes, évitez les réponses génériques, et restez concis."
)

def to_llama31_chat(system: str, user: str, assistant: str) -> str:
    return (
        "<|begin_of_text|>\n"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{system}\n"
        "<|eot_id|>\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{user}\n"
        "<|eot_id|>\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{assistant}\n"
        "<|eot_id|>\n"
    )

In [7]:
def faq_to_jsonl_llama31_text(faq_text, output_path="/content/drive/MyDrive/data_projet_llm/data/faq/faq_finetune.jsonl"):
    pairs = extract_qa_pairs(faq_text)

    with open(output_path, "w", encoding="utf-8") as f:
        for qnum, q, a in pairs:
            entry = {"text": to_llama31_chat(SYSTEM, q, a)}
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print("JSONL généré :", output_path)
    print("Nombre d'entrées :", len(pairs))
    return pairs

In [8]:
pairs = faq_to_jsonl_llama31_text(faq_text)

# vérifier la 1ère entrée
with open("/content/drive/MyDrive/data_projet_llm/data/faq/faq_finetune.jsonl", "r", encoding="utf-8") as f:
    first = json.loads(next(f))
print(first["text"][:400])

JSONL généré : /content/drive/MyDrive/data_projet_llm/data/faq/faq_finetune.jsonl
Nombre d'entrées : 30
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Vous êtes l’assistant officiel de l’Hôtel De la Promenade. Répondez en français, avec un ton chaleureux, élégant et professionnel. Donnez des informations exactes, évitez les réponses génériques, et restez concis.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
À quelle heure puis-je m'enregistrer à l'hôtel ?
<|eot_id|>
<|start_heade


# Installation des dependences et test du modele de base

In [9]:
%%capture
# Installation des dépendances
import os, importlib.util
! pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [10]:
# Charger le modele de base
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [11]:
PROMPT = """
Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des informations exactes.
4. N’inventez aucun tarif, horaire, service ou condition.
5. Si une information ne figure pas dans la FAQ, dites clairement :
   "Je ne retrouve pas cette information dans notre FAQ.
    Je vous invite à contacter la réception afin de vérifier ce point."

Réponse en 2 à 5 phrases maximum.
"""

In [12]:
# Definir le format de chat

def make_prompt(question):
    return f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{PROMPT}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [13]:
FastLanguageModel.for_inference(base_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSN

In [14]:
# Tester le modele de base
questions = [
    "À quelle heure puis-je m'enregistrer à l'hôtel ?",
    "Puis-je amener mon chien à l'hôtel ?",
    "Est-ce que votre hôtel offre le Wi-Fi gratuit ?",
]

for q in questions:

    print("QUESTION:", q)
    print("\n")
    
    inputs = tokenizer(make_prompt(q), return_tensors="pt").to("cuda")

    output = base_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(response)

QUESTION: À quelle heure puis-je m'enregistrer à l'hôtel ?



system

Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des informations exactes.
4. N’inventez aucun tarif, horaire, service ou condition.
5. Si une information ne figure pas dans la FAQ, dites clairement :
   "Je ne retrouve pas cette information dans notre FAQ.
    Je vous invite à contacter la réception afin de vérifier ce point."

Réponse en 2 à 5 phrases maximum.


user
À quelle heure puis-je m'enregistrer à l'hôtel?

assistant
Cher client, le processus d'enregistrement se déroule du lundi au dimanche, de 14h00 à 20h00. Pour toute arrivée prévue en dehors de ces horaires, nous vous recommandons de contacter la réception à l'avance afin de prendre les dispositions nécessaires.
QUESTION: Puis-je amener mon chien à l'hôtel ?



system

Vous êtes l’assist

In [15]:
# Sauvegarder les resultats du modele de base
base_outputs = []

for q in questions:
    inputs = tokenizer(make_prompt(q), return_tensors="pt").to("cuda")
    output = base_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    base_outputs.append({
        "question": q,
        "base_response": response
    })

# Entrainement Finetuning et test

In [16]:
# Definition du modele, des arguments de Lora et les parametres de l'entrainement
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Definition du modele
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

ft_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Les arguments de LoRA
model = FastLanguageModel.get_peft_model(
    ft_model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,  
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Recuperer le dataset
train_path = "/content/drive/MyDrive/data_projet_llm/data/faq/faq_finetune.jsonl"
dataset = load_dataset("json", data_files=train_path, split="train")

# Definir les parametres de l'entrainement
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,  
    args=TrainingArguments(
        output_dir="outputs_hotel",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,       
        learning_rate=8e-5,       
        warmup_steps=3,           
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.02,        
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        seed=42,
    ),
)
# Passer au train
trainer.train()

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
1,2.942300
2,3.000300
3,2.872800
4,3.036900
5,2.768900
6,2.436500
7,2.507700
8,2.298700
9,2.259400
10,2.300600


TrainOutput(global_step=12, training_loss=2.5742186109224954, metrics={'train_runtime': 96.421, 'train_samples_per_second': 0.933, 'train_steps_per_second': 0.124, 'total_flos': 1026237804503040.0, 'train_loss': 2.5742186109224954, 'epoch': 3.0})

In [17]:
from unsloth import FastLanguageModel
# Format de chat
def make_prompt(question):
    return f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{PROMPT}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [18]:
import torch

FastLanguageModel.for_inference(ft_model)  # <- model fine-tuné

def generate_answer(question, max_new_tokens=200):
    inputs = tokenizer(make_prompt(question), return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        # temperature=0.2,   # plus bas = plus fidèle
        top_p=0.9,
        do_sample=False,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [19]:
questions = [
   "À quelle heure puis-je m'enregistrer à l'hôtel ?",
    "Puis-je amener mon chien à l'hôtel ?",
    "Est-ce que votre hôtel offre le Wi-Fi gratuit ?",
]

for q in questions:
    print("QUESTION:", q)
    print("n")
    print(generate_answer(q))

QUESTION: À quelle heure puis-je m'enregistrer à l'hôtel ?
n

system

Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des informations exactes.
4. N’inventez aucun tarif, horaire, service ou condition.
5. Si une information ne figure pas dans la FAQ, dites clairement :
   "Je ne retrouve pas cette information dans notre FAQ.
    Je vous invite à contacter la réception afin de vérifier ce point."

Réponse en 2 à 5 phrases maximum.


user
À quelle heure puis-je m'enregistrer à l'hôtel?

assistant
L'enregistrement est possible à partir de 15h00.
QUESTION: Puis-je amener mon chien à l'hôtel ?
n

system

Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des

In [20]:
import json

finetuned_results = []
for q in questions:
    finetuned_results.append({
        "question": q,
        "finetuned_answer": generate_answer(q)
    })

with open("/content/finetuned_outputs.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, ensure_ascii=False, indent=2)

print("Sauvé: /content/finetuned_outputs.json")

Sauvé: /content/finetuned_outputs.json


# Comparer les deux modeles (Base vs Finetuning)

In [21]:
EVAL_SYSTEM = """
Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des informations exactes.
4. N’inventez aucun tarif, horaire, service ou condition.
5. Si une information ne figure pas dans la FAQ, dites clairement :
   "Je ne retrouve pas cette information dans notre FAQ.
    Je vous invite à contacter la réception afin de vérifier ce point."

Réponse en 2 à 5 phrases maximum.
"""

In [22]:
# Template Llama
def make_eval_prompt(question):
    return f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{EVAL_SYSTEM}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [23]:
# Parametre d'inference pour test comparatif
from unsloth import FastLanguageModel

def generate_eval(model, question, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(make_eval_prompt(question), return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        do_sample=False,          
        max_new_tokens=max_new_tokens,
        min_new_tokens=40,
        repetition_penalty=1.05
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [24]:
# Jeux de test comparatif
TEST_QUESTIONS = [
    # Questions exactes FAQ
    "À quelle heure puis-je m'enregistrer à l'hôtel ?",
    "Quel est le prix du stationnement ?",
    "À quelle heure est le départ ?",

    # Reformulations
    "Puis-je arriver vers midi ?",
    "Combien coûte le late check-out ?",
    "Le parking a-t-il une limite de hauteur ?",

    # Cas limite (risque hallucination)
    "Le parking est-il surveillé par caméras ?",
    "Avez-vous une piscine ?",
]

In [25]:
# Test des deux + sauvegarde
import json

def run_comparison(base_model, ft_model, questions, out_path="/content/drive/MyDrive/data_projet_llm/data/faq/compsre_outputs.json"):
    results = []
    for q in questions:
        base_ans = generate_eval(base_model, q)
        ft_ans   = generate_eval(ft_model, q)

        results.append({
            "question": q,
            "base_answer": base_ans,
            "finetuned_answer": ft_ans,
        })

        print("QUESTION:", q)
        print("==============================")
        print("\n--- BASE ---\n", base_ans)
        print("\n--- FINE-TUNED ---\n", ft_ans)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("\n Résultats sauvegardés :", out_path)
    return results

In [26]:
# Execution
results = run_comparison(
    base_model=base_model,
    ft_model=ft_model,   # ou ft_model=model si tu l’as nommé model
    questions=TEST_QUESTIONS,
    out_path="/content/drive/MyDrive/data_projet_llm/data/faq/compsre_outputs.json"
)

QUESTION: À quelle heure puis-je m'enregistrer à l'hôtel ?

--- BASE ---
 
system

Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

1. Répondez en français.
2. Adoptez un ton chaleureux, élégant et professionnel, fidèle à l’identité de l’hôtel.
3. Donnez uniquement des informations exactes.
4. N’inventez aucun tarif, horaire, service ou condition.
5. Si une information ne figure pas dans la FAQ, dites clairement :
   "Je ne retrouve pas cette information dans notre FAQ.
    Je vous invite à contacter la réception afin de vérifier ce point."

Réponse en 2 à 5 phrases maximum.


user
À quelle heure puis-je m'enregistrer à l'hôtel?

assistant
Bienvenue au Hôtel De la Promenade!

L'enregistrement est possible à partir de 15h00. Nous vous invitons à nous contacter pour confirmer votre arrivée et obtenir les informations de check-in détaillées.

--- FINE-TUNED ---
 
system

Vous êtes l’assistant officiel de l’Hôtel De la Promenade.

Instructions strictes :

In [27]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/data_projet_llm/data/faq/compare_outputs.csv", index=False, encoding="utf-8-sig")
df

,question,base_answer,finetuned_answer
0,À quelle heure puis-je m'enregistrer à l'hôtel ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
1,Quel est le prix du stationnement ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
2,À quelle heure est le départ ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
3,Puis-je arriver vers midi ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
4,Combien coûte le late check-out ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
5,Le parking a-t-il une limite de hauteur ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
6,Le parking est-il surveillé par caméras ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...
7,Avez-vous une piscine ?,\nsystem\n\nVous êtes l’assistant officiel de ...,\nsystem\n\nVous êtes l’assistant officiel de ...


In [28]:
import re
import pandas as pd

def extract_assistant(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # prend tout ce qui suit le dernier "assistant"
    m = re.split(r"\nassistant\n", text, maxsplit=0)
    if len(m) >= 2:
        ans = m[-1]
    else:
        # fallback : parfois tokens diffèrent
        ans = text
    return ans.strip()

# df = ton dataframe existant
df["base_clean"] = df["base_answer"].apply(extract_assistant)
df["finetuned_clean"] = df["finetuned_answer"].apply(extract_assistant)

df_clean = df[["question", "base_clean", "finetuned_clean"]].copy()
df_clean

,question,base_clean,finetuned_clean
0,À quelle heure puis-je m'enregistrer à l'hôtel ?,Bienvenue au Hôtel De la Promenade!\n\nL'enreg...,L'enregistrement est possible à partir de 15h0...
1,Quel est le prix du stationnement ?,Je ne retrouve pas cette information dans notr...,Je ne retrouve pas cette information dans notr...
2,À quelle heure est le départ ?,Je ne retrouve pas cette information dans notr...,Je ne retrouve pas cette information dans notr...
3,Puis-je arriver vers midi ?,Bien sûr! L'arrivée est possible à tout moment...,Bien sûr! L'arrivée est possible à tout moment...
4,Combien coûte le late check-out ?,Je ne retrouve pas cette information dans notr...,Je ne retrouve pas cette information dans notr...
5,Le parking a-t-il une limite de hauteur ?,Bien sûr! Notre parking dispose d'une hauteur ...,"Oui, le parking dispose d'une limite de hauteu..."
6,Le parking est-il surveillé par caméras ?,Bien sûr! Notre parking est équipé de caméras ...,"Oui, le parking de l'Hôtel De la Promenade est..."
7,Avez-vous une piscine ?,"Oui, notre hôtel dispose d'une piscine intérie...","Oui, notre hôtel dispose d'une piscine intérie..."


In [29]:
for i, row in df_clean.iterrows():
    print("\n" + "="*80)
    print("QUESTION :")
    print(row["question"])
    print("\n--- BASE MODEL ---")
    print(row["base_clean"])
    print("\n--- FINE-TUNED MODEL ---")
    print(row["finetuned_clean"])
    print("="*80)


QUESTION :
À quelle heure puis-je m'enregistrer à l'hôtel ?

--- BASE MODEL ---
Bienvenue au Hôtel De la Promenade!

L'enregistrement est possible à partir de 15h00. Nous vous invitons à nous contacter pour confirmer votre arrivée et obtenir les informations de check-in détaillées.

--- FINE-TUNED MODEL ---
L'enregistrement est possible à partir de 15h00. Veuillez noter que nous avons un système d'enregistrement anticipé pour les clients qui arrivent tôt. N'hésitez pas à nous contacter si vous avez besoin de plus d'informations.

QUESTION :
Quel est le prix du stationnement ?

--- BASE MODEL ---
Je ne retrouve pas cette information dans notre FAQ. Je vous invite à contacter la réception afin de vérifier ce point. Nous sommes à votre disposition pour répondre à vos questions et vous aider à planifier votre séjour.

--- FINE-TUNED MODEL ---
Je ne retrouve pas cette information dans notre FAQ. Je vous invite à contacter la réception afin de vérifier ce point. Nous sommes là pour vous aid

# Analyse comparative des modèles : Base vs Fine-Tuned

## 1. Contexte de l’évaluation

L’objectif de cette évaluation est de comparer les performances du modèle de base (*Meta-Llama-3.1-8B-Instruct*) et du modèle fine-tuné (LoRA entraîné sur la FAQ de l’Hôtel De la Promenade) à partir des réponses générées sur un ensemble de questions de test.

Les critères d’analyse retenus sont :

- Exactitude factuelle (respect strict des informations de la FAQ)
- Présence ou absence d’hallucination
- Cohérence stylistique avec l’identité de l’hôtel
- Capacité de généralisation sur des reformulations
- Gestion des questions non couvertes par la FAQ

Les réponses ont été générées avec des paramètres déterministes (`do_sample=False`) afin d’assurer une comparaison équitable et reproductible.

---

## 2. Analyse qualitative détaillée

### 2.1 Question factuelle directe : heure d’enregistrement

Les deux modèles indiquent correctement que l’enregistrement est possible à partir de 15h00.

Le modèle de base fournit une réponse correcte et relativement neutre.  
Le modèle fine-tuné propose une formulation plus concise et légèrement plus structurée, mais introduit une mention vague d’un système d’enregistrement anticipé sans précision tarifaire ou conditionnelle claire.

Interprétation :  
Le fine-tuning améliore la forme et la concision, mais ne garantit pas une stricte fidélité aux formulations exactes de la FAQ.

---

### 2.2 Informations tarifaires (stationnement, départ, late check-out)

Pour les questions concernant :
- le prix du stationnement,
- l’heure de départ,
- le coût du late check-out,

les deux modèles répondent qu’ils ne retrouvent pas l’information dans la FAQ.

Or, ces informations figurent dans le document d’entraînement.

Cela suggère :
- une mémorisation partielle des données,
- une capacité limitée de rappel précis,
- ou un effet de prudence excessive induit par le prompt strict.

Interprétation :  
Le fine-tuning n’a pas permis une consolidation robuste des détails tarifaires. L’alignement stylistique semble plus marqué que l’ancrage factuel.

---

### 2.3 Reformulation : arrivée vers midi

À la question « Puis-je arriver vers midi ? », les deux modèles répondent que l’arrivée est possible à tout moment.

Cette réponse est incorrecte au regard de la FAQ.

Il s’agit d’une hallucination typique :
- Le modèle généraliste applique une règle hôtelière générique.
- Le modèle fine-tuné ne corrige pas ce biais pré-entraîné.

Interprétation :  
Le fine-tuning par LoRA (environ 0,17 % des paramètres entraînables) ne modifie pas suffisamment les représentations internes pour neutraliser les comportements probabilistes appris lors du pré-entraînement.

---

### 2.4 Détail technique précis : hauteur du parking

Les deux modèles indiquent correctement une limite de hauteur de 2,10 mètres.

Le modèle fine-tuné se montre plus concis et légèrement mieux structuré.

Interprétation :  
Lorsque l’information est numérique, explicite et clairement représentée dans le dataset, le fine-tuning permet une restitution fidèle tout en améliorant la qualité rédactionnelle.

---

### 2.5 Questions non couvertes par la FAQ (caméras, piscine)

Pour les questions :
- « Le parking est-il surveillé par caméras ? »
- « Avez-vous une piscine ? »

les deux modèles inventent des réponses positives.

Le modèle fine-tuné ajoute parfois davantage de détails (par exemple des précisions sur la conservation des images ou des éléments descriptifs supplémentaires).

Interprétation :  
Le fine-tuning stylistique améliore la fluidité et la cohérence narrative, mais ne supprime pas les mécanismes génératifs probabilistes du modèle de base. Les hallucinations persistent et peuvent même être amplifiées par une rédaction plus détaillée.

---

## 3. Synthèse comparative

| Critère | Modèle de base | Modèle fine-tuné |
|----------|----------------|------------------|
| Style et ton | Correct mais générique | Plus aligné avec l’identité |
| Exactitude factuelle | Variable | Légèrement améliorée sur certains points |
| Rappel des tarifs | Incomplet | Incomplet |
| Hallucinations | Présentes | Toujours présentes |
| Gestion hors FAQ | Répond au lieu de refuser | Répond au lieu de refuser |

---

## 4. Interprétation technique

Le fine-tuning par LoRA agit principalement comme un mécanisme d’alignement stylistique plutôt que comme une reconfiguration profonde des connaissances internes.

Compte tenu :
- de la petite taille du dataset (30 exemples),
- du faible pourcentage de paramètres entraînables,
- de l’absence de mécanisme de retrieval externe,

le modèle conserve une forte dépendance aux distributions probabilistes générales issues du pré-entraînement.

Ainsi :
- le ton et la cohérence rédactionnelle s’améliorent,
- la concision progresse,
- mais la fidélité factuelle n’est pas garantie.

---

## 5. Conclusion

Cette comparaison montre que le fine-tuning léger par LoRA permet d’adapter efficacement le style et l’identité de marque d’un assistant conversationnel.

Cependant, il ne suffit pas à garantir une stricte fidélité aux données sources ni à éliminer les hallucinations.

Pour un déploiement en contexte réel, une architecture combinant fine-tuning stylistique et Retrieval-Augmented Generation (RAG) serait plus adaptée afin d’assurer un contrôle fiable de la véracité des réponses.

En conclusion, le fine-tuning améliore l’alignement et la cohérence stylistique, mais ne constitue pas à lui seul un mécanisme robuste de validation factuelle.